In [0]:
pip install xgboost

In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq

In [0]:
lgbm_df = spark.read.table("hive_metastore.default.lreg_df")

#### LGBM

In [0]:
final_df = lgbm_df.drop("AO","NOME","TIPOINST","TAGCOM","REDE","temperature","humidity","precipitation","wind_speed","ID_prefix","ID_OBJECTO")

In [0]:
indexers = [
    StringIndexer(inputCol="ID", outputCol="ID_idx", handleInvalid="keep"),
    StringIndexer(inputCol="AM_PM", outputCol="AM_PM_idx", handleInvalid="keep"),
    StringIndexer(inputCol="CONCELHO", outputCol="CONCELHO_idx", handleInvalid="keep")
]

# Fit and transform sequentially
df_transformed = final_df
for indexer in indexers:
    model = indexer.fit(df_transformed)
    df_transformed = model.transform(df_transformed)

In [0]:
train = df_transformed.filter(col("DATE") < "2023-11-30")
test  = df_transformed.filter(col("DATE") >= "2023-11-30")

In [0]:
feature_cols = [
    "INTENSITY", "TENSION", "H_LIM_I", "H_LIM_T",
    "MAVERAGE_2H_I", "MAVERAGE_2H_T", "MAVERAGE_1D_I", "MAVERAGE_1D_T",
    "EVENT_COUNT_I", "EVENT_COUNT_T", "TIME_OVER_LIMIT_I", "TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK", "DAY_OF_MONTH", "DAY_OF_YEAR", "HOUR_OF_DAY",
    "ID_idx", "AM_PM_idx", "CONCELHO_idx"
]

train_pd = train.select(feature_cols + ["has_falha"]).toPandas()
test_pd  = test.select(feature_cols + ["has_falha"]).toPandas()

In [0]:
X_train = train_pd[feature_cols]
y_train = train_pd["has_falha"]

X_test = test_pd[feature_cols]
y_test = test_pd["has_falha"]


In [0]:


model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    use_label_encoder=False
)
model.fit(X_train, y_train)

_ = evaluate_estimator_binary(model, X_test, y_test,
                              model_name="xgb_baseline",
                              base_dir="/dbfs/reports")


In [0]:
model.fit(X_train, y_train)

#### Analysis

In [0]:
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

In [0]:
import matplotlib.pyplot as plt
xgb.plot_importance(model, max_num_features=20)
plt.show()

In [0]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix
)

def _metrics_at_threshold(y_true, y_prob, thr: float):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tp + tn + fp + fn
    acc   = (tp + tn) / total if total else float('nan')
    prec  = tp / (tp + fp) if (tp + fp) else 0.0
    rec   = tp / (tp + fn) if (tp + fn) else 0.0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    spec  = tn / (tn + fp) if (tn + fp) else 0.0
    bal   = (rec + spec) / 2
    denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc   = ((tp*tn - fp*fn) / denom) if denom else 0.0
    return dict(thr=float(thr), tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
                acc=acc, prec=prec, rec=rec, f1=f1, spec=spec, bal_acc=bal, mcc=mcc)

def _save_cm(cm_dict, title, out_png):
    cm = np.array([[cm_dict['tn'], cm_dict['fp']],
                   [cm_dict['fn'], cm_dict['tp']]], dtype=int)
    fig, ax = plt.subplots(figsize=(4, 4), dpi=160)
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['0', '1'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['0', '1'])
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches='tight'); plt.close(fig)

def evaluate_estimator_binary(estimator, X_test, y_test,
                              model_name="model_v1",
                              base_dir="/dbfs/reports"):
    """
    Works with any sklearn-like estimator that has predict_proba(X)[:,1].
    Saves artifacts and prints a compact summary.
    """
    os.makedirs(base_dir, exist_ok=True)
    outdir = os.path.join(base_dir, model_name)
    os.makedirs(outdir, exist_ok=True)

    # --- Probabilities
    y_prob = estimator.predict_proba(X_test)[:, 1]
    y_true = np.asarray(y_test).astype(int)

    # --- Global metrics
    try: auc = roc_auc_score(y_true, y_prob)
    except: auc = float('nan')
    try: ap  = average_precision_score(y_true, y_prob)
    except: ap  = float('nan')

    # --- Thresholded metrics
    m05 = _metrics_at_threshold(y_true, y_prob, 0.5)

    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12) if thr.size else np.array([])
    best_idx = int(np.nanargmax(f1s)) if f1s.size else 0
    best_thr = float(thr[best_idx]) if thr.size else 0.5
    mbest = _metrics_at_threshold(y_true, y_prob, best_thr)

    # --- Plots: Confusion Matrices
    _save_cm(m05, f'Confusion Matrix @0.5\nAcc {m05["acc"]:.3f} F1 {m05["f1"]:.3f}',
             os.path.join(outdir, "cm_thr0.5.png"))
    _save_cm(mbest, f'Confusion Matrix @bestF1={mbest["thr"]:.3f}\nAcc {mbest["acc"]:.3f} F1 {mbest["f1"]:.3f}',
             os.path.join(outdir, "cm_bestf1.png"))

    # --- Plots: ROC & PR
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig = plt.figure(figsize=(5, 4), dpi=160); ax = plt.gca()
    ax.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], '--', linewidth=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC'); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "roc.png"), bbox_inches='tight'); plt.close(fig)

    fig = plt.figure(figsize=(5, 4), dpi=160); ax = plt.gca()
    ax.plot(rec, prec, label=f'AP={ap:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR curve'); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(outdir, "pr.png"), bbox_inches='tight'); plt.close(fig)

    # --- Save summary
    summary = dict(auc=float(auc), ap=float(ap), thr0_5=m05, best_f1=mbest)
    with open(os.path.join(outdir, "metrics.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # --- Console summary
    print(f"[{model_name}] AUC={auc:.4f} AP={ap:.4f}")
    print(f"  @0.5                : acc={m05['acc']:.3f} prec={m05['prec']:.3f} rec={m05['rec']:.3f} "
          f"f1={m05['f1']:.3f} spec={m05['spec']:.3f} mcc={m05['mcc']:.3f}")
    print(f"  @bestF1 (thr={mbest['thr']:.3f}): acc={mbest['acc']:.3f} prec={mbest['prec']:.3f} rec={mbest['rec']:.3f} "
          f"f1={mbest['f1']:.3f} spec={mbest['spec']:.3f} mcc={mbest['mcc']:.3f}")
    print(f"Artifacts written to: {outdir}")

    return summary


In [0]:
print_saved_report("xgb_baseline")
# or: print_saved_report("xgb_baseline")


In [0]:
import os, json
from IPython.display import display
from PIL import Image

model_name = "xgb_baseline"   # <- change to your run name
outdir = f"/dbfs/reports/{model_name}"

# 1) Load & print metrics
with open(os.path.join(outdir, "metrics.json"), "r") as f:
    M = json.load(f)

print(f"[{model_name}]")
print(f"AUC: {M['auc']:.4f} | AP: {M['ap']:.4f}")
m05 = M["thr0_5"]; mb = M["best_f1"]
print(" @0.5      -> "
      f"acc={m05['acc']:.3f} prec={m05['prec']:.3f} rec={m05['rec']:.3f} "
      f"f1={m05['f1']:.3f} spec={m05['spec']:.3f} mcc={m05['mcc']:.3f}")
print(f" @bestF1({mb['thr']:.3f}) -> "
      f"acc={mb['acc']:.3f} prec={mb['prec']:.3f} rec={mb['rec']:.3f} "
      f"f1={mb['f1']:.3f} spec={mb['spec']:.3f} mcc={mb['mcc']:.3f}")

# 2) Display figures
for png in ["cm_thr0.5.png", "cm_bestf1.png", "roc.png", "pr.png"]:
    path = os.path.join(outdir, png)
    print(png)
    display(Image.open(path))


In [0]:
import os, json, builtins
from datetime import datetime

def print_saved_report(model_name, base_dir="/dbfs/reports"):
    path = f"{base_dir}/{model_name}/metrics.json"
    with open(path, "r") as f:
        M = json.load(f)

    ts = datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y%m%d_%H%M%S")

    # N and positive rate from the 0.5 confusion counts
    t = M.get("thr0_5", {})
    tp, fp, tn, fn = (t.get("tp", 0), t.get("fp", 0), t.get("tn", 0), t.get("fn", 0))
    N = tp + tn + fp + fn
    pos_rate = (tp + fn) / (builtins.max(N, 1))

    def fmt(x, nd=4):
        try: return f"{x:.{nd}f}"
        except: return "nan"

    print(f"=== {model_name} | {ts} ===")
    print(f"N={N}, Pos rate={fmt(pos_rate)}")
    print(f"AUC={fmt(M.get('auc', float('nan')))}, AP={fmt(M.get('ap', float('nan')))}")

    if "thr0_5" in M:
        m05 = M["thr0_5"]
        print(f"[thr=0.50]  ACC={fmt(m05.get('acc', float('nan')))}  "
              f"PREC={fmt(m05.get('prec', float('nan')))}  "
              f"REC={fmt(m05.get('rec', float('nan')))}  "
              f"F1={fmt(m05.get('f1', float('nan')))}")

    if "best_f1" in M:
        mb = M["best_f1"]
        thr = mb.get("thr", float("nan"))
        print(f"[thr={thr:.3f}] ACC={fmt(mb.get('acc', float('nan')))}  "
              f"PREC={fmt(mb.get('prec', float('nan')))}  "
              f"REC={fmt(mb.get('rec', float('nan')))}  "
              f"F1={fmt(mb.get('f1', float('nan')))}")
